# 8 面向对象高级编程

## 8.1 __slots__

Python 每个实例都有一个字典 __dict__ 来存储所有的属性,如果需要创建数百万个小对象，这些字典积累起来的内存开销是非常惊人的,__slots__ 的作用就是：

限制：告诉 Python 这个类只能有这几个属性

省内存：不再使用 __dict__ 存储属性，从而大幅降低内存消耗

In [1]:
class Student(object):
    __slots__ = ('name', 'age') # Limit attributes to 'name' and 'age' only

s = Student()
s.name = 'Alice'
s.age = 20

try:
    s.grade = 'A'  # This will raise an AttributeError
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'Student' object has no attribute 'grade'


__slots__ 定义的属性仅对当前类实例起作用，对继承的子类是不起作用的

## 8.2 @property

Python内置的@property装饰器负责把一个方法变成属性调用

In [2]:
class Student(object):
    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if not isinstance(value, int):
            raise ValueError('score must be an integer!')
        if value < 0 or value > 100:
            raise ValueError('score must between 0 ~ 100!')
        self._score = value


内部变量名和属性名不能写的一模一样，在方法内部，一定要用 self._score（带下划线）来存储实际数值

Practice: 利用@property给一个Screen对象加上width和height属性，以及一个只读属性resolution

In [3]:
class Screen(object):
    @property
    def width(self):
        return self._width

    @width.setter
    def width(self, value):
        if not isinstance(value, int):
            raise ValueError('width must be an integer!')
        self._width = value
    
    @property
    def height(self):
        return self._height
    
    @height.setter
    def height(self, value):
        if not isinstance(value, int):
            raise ValueError('height must be an integer!')
        self._height = value
    
    @property
    def resolution(self):
        return self._width * self._height

s = Screen()
s.width = 1024
s.height = 768
print('width =', s.width)
print('height =', s.height)
print('resolution =', s.resolution)

if s.resolution == 786432:
    print('测试通过!')
else:
    print('测试失败!')

width = 1024
height = 768
resolution = 786432
测试通过!


## 8.3 多重继承

继承是父类 -> 子类的一对一关系，多重继承允许一个子类同时继承多个父类，从而同时获得多个父类的所有功能

In [4]:
class Animal(object):
    pass

# 大类
class Mammal(Animal):
    pass

class Bird(Animal):
    pass

# 具体类 (Mix-in)
class Runnable(object):
    def run(self):
        print('Running...')

class Flyable(object):
    def fly(self):
        print('Flying...')

# 具体动物类
class Dog(Mammal, Runnable):
    pass
class Bat(Mammal, Flyable):
    pass
class Parrot(Bird, Flyable):
    pass

## 8.4 定制类

str & repr: 美化打印

In [5]:
class Student(object):
    def __init__(self, name):
        self.name = name
    def __str__(self): # string representation
        return 'Student object (name: %s)' % self.name
    __repr__ = __str__ # official representation
print(Student('Bob'))

Student object (name: Bob)


iter & next: 让类可迭代

In [6]:
class Fib(object):
    def __init__(self):
        self.a, self.b = 0, 1  # 初始化两个计数器a，b

    def __iter__(self): # 实例本身就是迭代对象，故返回自己
        return self
    
    def __next__(self):
        self.a, self.b = self.b, self.a + self.b

        if self.a > 100000:
            raise StopIteration()
        return self.a

for n in Fib():
    print(n)

1
1
2
3
5
8
13
21
34
55
89
144
233
377
610
987
1597
2584
4181
6765
10946
17711
28657
46368
75025


getitem: 取索引

In [7]:
class Fib(object):
    def __getitem__(self, n):
        a, b = 0, 1
        for x in range(n):
            a, b = b, a + b
        return a
print(Fib()[0])  # 0
print(Fib()[1])  # 1

0
1


getattr: 动态处理属性

In [8]:
class Student(object):
    def __init__(self):
        self.name = 'Michael'

    def __getattr__(self, attr): # only called when attribute not found the usual ways
        if attr=='score':
            return 99

s = Student()
print(s.name)    # Michael
print(s.score)   # 99

Michael
99


call: 在实例本身上调用属性和方法

In [9]:
class Student(object):
    def __init__(self,name):
        self.name = name
    
    def __call__(self):
        print('My name is %s.' % self.name)
s = Student('Michael')
s()  # My name is Michael.

My name is Michael.


## 8.5 枚举类

枚举成员的值一旦定义，就不能在外部修改

枚举类是可迭代的，可以轻松列出所有成员

枚举成员可以用 is 或 == 进行比较

In [10]:
from enum import Enum

class Weekday(Enum):
    Sun = 0
    Mon = 1
    Tue = 2
    Wed = 3
    Thu = 4
    Fri = 5
    Sat = 6

day1 = Weekday.Mon
print(day1)          # Weekday.Mon
print(day1.name)   # Mon
print(day1.value)  # 1
print(Weekday.Tue)  # Weekday.Tue

for day in Weekday:
    print(day)

Weekday.Mon
Mon
1
Weekday.Tue
Weekday.Sun
Weekday.Mon
Weekday.Tue
Weekday.Wed
Weekday.Thu
Weekday.Fri
Weekday.Sat


还能限制重复值/自动赋值（各不相同）

In [11]:
from enum import Enum, unique, auto

@unique
class Student(Enum):
    Michael = 1
    Bob = 2
    #Tracy = 1 # This will raise an error

class Color(Enum):
    Red = auto()
    Green = auto()
    Blue = auto()

## 8.6 元类

type 不仅可以返回对象的类，也可以创建类

In [12]:
def fn(self, name='world'):
    print('Hello, %s.' % name)

Hello = type('Hello', (object,), dict(hello=fn)) # 参数：1.类名 2.父类集合(tuple) 3.方法名与函数的绑定(dict)

h = Hello()
h.hello()  # Hello, world.
h.hello('Kan')  # Hello, Kan.

Hello, world.
Hello, Kan.


实例是由类创建的,那么类就是由元类创建的

定义一个元类需要遵循以下规则：

继承 type：元类必须是 type 的子类

重写 __new__ 方法：这是在类创建前执行的方法

In [ ]:
# 1. 定义元类：习惯以 Metaclass 结尾
class ListMetaclass(type):
    def __new__(cls, name, bases, attrs):
        # 给类自动添加一个 add 方法
        attrs['add'] = lambda self, value: self.append(value)
        return type.__new__(cls, name, bases, attrs)

# 2. 使用元类：通过 metaclass 关键字
class MyList(list, metaclass=ListMetaclass):
    pass

L = MyList()
L.add(1) # MyList 本来没有 add，是元类动态加进去的
print(L) # 输出: [1]